# Робочий процес з людиною в циклі за допомогою Microsoft Agent Framework

## 🎯 Цілі навчання

У цьому ноутбуці ви навчитеся впроваджувати робочі процеси **людина в циклі** за допомогою `RequestInfoExecutor` у Microsoft Agent Framework. Цей потужний шаблон дає змогу призупиняти AI-робочі процеси для збору людського вводу, роблячи ваших агентів інтерактивними та даючи людям контроль над критичними рішеннями.

## 🔄 Що таке людина в циклі?

**Людина в циклі (HITL)** — це шаблон проєктування, коли AI-агенти призупиняють виконання, щоб запросити людський ввід перед продовженням. Це необхідно для:

- ✅ **Критичних рішень** — отримати людське схвалення перед важливими діями
- ✅ **Невизначених ситуацій** — дати людям уточнити, коли AI не впевнений
- ✅ **Переваг користувачів** — попросити користувачів обрати між кількома варіантами
- ✅ **Відповідності та безпеки** — забезпечити людський нагляд за регульованими операціями
- ✅ **Інтерактивних досвідів** — будувати розмовних агентів, які реагують на ввід користувача

## 🏗️ Як це працює в Microsoft Agent Framework

Фреймворк надає три ключові компоненти для HITL:

1. **`RequestInfoExecutor`** — спеціальний виконавець, який призупиняє робочий процес і генерує `RequestInfoEvent`
2. **`RequestInfoMessage`** — базовий клас для типізованих запитів, що надсилаються людям
3. **`RequestResponse`** — корелює людські відповіді з початковими запитами за допомогою `request_id`

**Шаблон робочого процесу:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Наш приклад: бронювання готелю з підтвердженням користувача

Ми розширимо умовний робочий процес, додавши людське підтвердження **перед** пропозицією альтернативних напрямків:

1. Користувач запитує напрямок (наприклад, «Париж»)
2. `availability_agent` перевіряє наявність номерів
3. **Якщо немає номерів** → `confirmation_agent` запитує «Бажаєте побачити альтернативи?»
4. Робочий процес **призупиняється** за допомогою `RequestInfoExecutor`
5. **Людина відповідає** «так» або «ні» через консоль
6. `decision_manager` маршрутизує на основі відповіді:
   - **Так** → Показати альтернативні напрямки
   - **Ні** → Скасувати запит на бронювання
7. Відобразити кінцевий результат

Цей приклад демонструє, як надати користувачам контроль над пропозиціями агента!

---

Розпочнемо! 🚀


## Крок 1: Імпорт потрібних бібліотек

Ми імпортуємо стандартні компоненти Agent Framework, а також **класи, специфічні для людського втручання**:
- `RequestInfoExecutor` - Виконавець, який призупиняє робочий процес для введення людиною
- `RequestInfoEvent` - Подія, що виникає, коли запитується введення людиною
- `RequestInfoMessage` - Базовий клас для типізованих payload-запитів
- `RequestResponse` - Корелює людські відповіді з запитами
- `WorkflowOutputEvent` - Подія для виявлення вихідних даних робочого процесу


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Крок 2: Визначте моделі Pydantic для структурованих виходів

Ці моделі визначають **схему**, яку агенти повертатимуть. Ми зберігаємо всі моделі з умовного робочого процесу та додаємо:

**Нове для Human-in-the-Loop:**
- `HumanFeedbackRequest` - підклас `RequestInfoMessage`, який визначає дані запиту, що надсилаються людям
  - Містить `prompt` (питання для запиту) та `destination` (контекст щодо недоступного міста)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Крок 3: Створіть інструмент бронювання готелю

Той самий інструмент із умовного робочого процесу - перевіряє, чи є вільні номери у пункті призначення.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Крок 4: Визначення функцій умов для маршрутизації

Нам потрібно **чотири функції умов** для нашого робочого процесу з участю людини:

**З умовного робочого процесу:**
1. `has_availability_condition` - Виконує маршрутизацію, коли готелі ДОСТУПНІ
2. `no_availability_condition` - Виконує маршрутизацію, коли готелі НЕДОСТУПНІ

**Нові для робочого процесу з участю людини:**
3. `user_wants_alternatives_condition` - Виконує маршрутизацію, коли користувач каже "так" альтернативам
4. `user_declines_alternatives_condition` - Виконує маршрутизацію, коли користувач каже "ні" альтернативам


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Крок 5: Створіть виконавець Decision Manager

Це **ядро патерну з участю людини у циклі**! `DecisionManager` — це власний `Executor`, який:

1. **Отримує відгук від людини** через об’єкти `RequestResponse`
2. **Опрацьовує рішення користувача** (так/ні)
3. **Маршрутизує робочий процес**, надсилаючи повідомлення відповідним агентам

Ключові особливості:
- Використовує декоратор `@handler` для оголошення методів як кроків робочого процесу
- Отримує `RequestResponse[HumanFeedbackRequest, str]`, який містить і початковий запит, і відповідь користувача
- Генерує прості повідомлення "так" або "ні", які запускають наші функції умов


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Крок 6: Створення власного виконавця відображення

Той самий виконавець відображення з умовного робочого процесу - повертає кінцеві результати як вихідні дані робочого процесу.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Крок 7: Завантаження змінних середовища

Налаштуйте клієнт LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Крок 8: Створення агентів штучного інтелекту та виконавців

Ми створюємо **шість компонентів робочого процесу**:

**Агенти (обгорнуті в AgentExecutor):**
1. **availability_agent** - Перевіряє наявність готелю за допомогою інструменту
2. **confirmation_agent** - 🆕 Готує запит на підтвердження людиною
3. **alternative_agent** - Пропонує альтернативні міста (коли користувач каже так)
4. **booking_agent** - Заохочує бронювання (коли є доступні кімнати)
5. **cancellation_agent** - 🆕 Обробляє повідомлення про скасування (коли користувач каже ні)

**Спеціальні виконавці:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, який призупиняє робочий процес для введення людиною
7. **decision_manager** - 🆕 Спеціальний виконавець, який маршрутизує на основі відповіді людини (вже визначений вище)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Крок 9: Створення робочого процесу із людським контролем

Тепер ми створюємо граф робочого процесу з **умовною маршрутизацією**, включаючи шлях із людським контролем:

**Структура робочого процесу:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Ключові ребра:**
- `availability_agent → confirmation_agent` (коли немає кімнат)
- `confirmation_agent → prepare_human_request` (трансформація типу)
- `prepare_human_request → request_info_executor` (пауза для людини)
- `request_info_executor → decision_manager` (завжди - надає RequestResponse)
- `decision_manager → alternative_agent` (коли користувач каже "так")
- `decision_manager → cancellation_agent` (коли користувач каже "ні")
- `availability_agent → booking_agent` (коли кімнати доступні)
- Всі шляхи закінчуються на `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Крок 10: Запустіть тестовий випадок 1 – Місто БЕЗ доступності (Паризь з підтвердженням людини)

Цей тест демонструє **повний цикл участі людини в процесі**:

1. Запит готелю в Парижі
2. availability_agent перевіряє → Немає кімнат
3. confirmation_agent створює питання для людини
4. request_info_executor **призупиняє робочий процес** і випускає `RequestInfoEvent`
5. **Додаток виявляє подію і виводить запит користувачу в консолі**
6. Користувач вводить "yes" або "no"
7. Додаток надсилає відповідь через `send_responses_streaming()`
8. decision_manager маршрутизує на основі відповіді
9. Відображається кінцевий результат

**Ключовий патерн:**
- Використовуйте `workflow.run_stream()` для першої ітерації
- Використовуйте `workflow.send_responses_streaming(pending_responses)` для наступних ітерацій
- Слухайте `RequestInfoEvent`, щоб визначити, коли потрібен ввід людини
- Слухайте `WorkflowOutputEvent`, щоб отримати кінцеві результати


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Крок 11: Запустіть тестовий випадок 2 - Місто З Наявністю (Стокгольм - Без Потрібного Людського Вводу)

Цей тест демонструє **прямий шлях**, коли кімнати доступні:

1. Запит готелю у Стокгольмі
2. availability_agent перевіряє → Кімнати доступні ✅
3. booking_agent пропонує бронювання
4. display_result показує підтвердження
5. **Людський ввід не потрібен!**

Робочий процес повністю обходить шлях з людським втручанням, коли кімнати доступні.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Основні Висновки та Найкращі Практики Людина-в-Циклі

### ✅ Що Ви Вивчили:

#### 1. **Патерн RequestInfoExecutor**
Патерн людина-в-циклі у Microsoft Agent Framework використовує три ключові компоненти:
- `RequestInfoExecutor` - призупиняє робочий процес і генерує події
- `RequestInfoMessage` - базовий клас для типізованих запитів (створюйте підкласи!)
- `RequestResponse` - корелює людські відповіді з оригінальними запитами

**Критичне розуміння:**
- `RequestInfoExecutor` сам не збирає ввід — він лише призупиняє робочий процес
- Ваш код додатку має слухати `RequestInfoEvent` та збирати ввід
- Ви маєте викликати `send_responses_streaming()` з словником, що мапить `request_id` на відповідь користувача

#### 2. **Патерн потокового виконання**
```python
# Перша ітерація
stream = workflow.run_stream(initial_request)

# Наступні ітерації (після введення користувача)
stream = workflow.send_responses_streaming(pending_responses)

# Завжди обробляти події
events = [event async for event in stream]
```

#### 3. **Архітектура на основі подій**
Слухайте конкретні події для контролю робочого процесу:
- `RequestInfoEvent` - потрібен людський ввід (робочий процес призупинено)
- `WorkflowOutputEvent` - доступний кінцевий результат (робочий процес завершено)
- `WorkflowStatusEvent` - зміни стану (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS тощо)

#### 4. **Користувацькі виконавці з @handler**
`DecisionManager` демонструє, як створювати виконавців, які:
- Використовують декоратор `@handler` для відкриття методів як кроків робочого процесу
- Приймають типізовані повідомлення (наприклад, `RequestResponse[HumanFeedbackRequest, str]`)
- Маршрутизують робочий процес, надсилаючи повідомлення іншим виконавцям
- Отримують доступ до контексту через `WorkflowContext`

#### 5. **Умовне Маршрутування з Людськими Рішеннями**
Ви можете створювати функції умов, які оцінюють людські відповіді:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Застосування у Реальному Житті:

1. **Процеси Затверджень**
   - Отримуйте затвердження менеджера перед обробкою звітів про витрати
   - Потрібен людський перегляд перед надсиланням автоматичних листів
   - Підтверджуйте високовартісні транзакції перед виконанням

2. **Модерація Контенту**
   - Позначайте сумнівний контент для людського перегляду
   - Просіть модераторів ухвалювати остаточне рішення у складних випадках
   - Ескалюйте до людей, коли AI має низьку впевненість

3. **Обслуговування Клієнтів**
   - Дозвольте AI автоматично обробляти стандартні питання
   - Ескалюйте складні питання до людських агентів
   - Запитайте клієнта, чи хоче він спілкуватися з людиною

4. **Обробка Даних**
   - Попросіть людей вирішувати неоднозначні дані
   - Підтверджуйте інтерпретації AI щодо нечітких документів
   - Дозвольте користувачам обирати між кількома допустимими інтерпретаціями

5. **Критичні для Безпеки Системи**
   - Потрібне людське підтвердження перед незворотними діями
   - Отримуйте затвердження перед доступом до чутливих даних
   - Підтверджуйте рішення у регульованих галузях (охорона здоров’я, фінанси)

6. **Інтерактивні Агенти**
   - Створюйте розмовних ботів, що задають уточнюючі питання
   - Розробляйте майстрів, які супроводжують користувачів через складні процеси
   - Проєктуйте агентів, які крок за кроком співпрацюють з людьми

### 🔄 Порівняння: З Людиною-в-Циклі vs Без Неї

| Особливість | Умовний Робочий Процес | Робочий Процес Людина-в-Циклі |
|---------|---------------------|---------------------------|
| **Виконання** | Один `workflow.run()` | Цикл з `run_stream()` + `send_responses_streaming()` |
| **Користувацький Ввід** | Відсутній (повністю автоматичний) | Інтерактивні підказки через `input()` або UI |
| **Компоненти** | Агенти + Виконавці | + RequestInfoExecutor + DecisionManager |
| **Події** | Лише AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent тощо |
| **Призупинка** | Без призупинки | Робочий процес призупиняється у RequestInfoExecutor |
| **Людський Контроль** | Відсутній | Люди приймають ключові рішення |
| **Випадок Використання** | Автоматизація | Співпраця та нагляд |

### 🚀 Розширені Патерни:

#### Багато Точок Прийняття Людських Рішень
Ви можете мати кілька вузлів `RequestInfoExecutor` у одному робочому процесі:
```python
.add_edge(agent1, request_info_1)  # Перше людське рішення
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Друге людське рішення
.add_edge(decision_manager_2, final_agent)
```

#### Обробка Тайм-аутів
Реалізуйте тайм-аути на людські відповіді:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # За замовчуванням безпечний варіант
```

#### Багате Інтегрування UI
Замість `input()` інтегруйте з веб-UI, Slack, Teams тощо:
```python
if isinstance(event, RequestInfoEvent):
    # Надіслати сповіщення на обраний користувачем канал
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Умовна Людина-в-Циклі
Запитуйте людський ввід лише у конкретних ситуаціях:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Направляти до людини лише якщо довіра низька або значення високе
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Найкращі Практики:

1. **Завжди Створюйте Підкласи RequestInfoMessage**
   - Забезпечує типову безпеку та валідацію
   - Дає багатий контекст для відображення UI
   - Прояснює призначення кожного типу запиту

2. **Використовуйте Описові Підказки**
   - Включайте контекст того, що ви запитуєте
   - Пояснюйте наслідки кожного вибору
   - Задавайте прості і зрозумілі питання

3. **Обробляйте Неочікуваний Ввід**
   - Перевіряйте відповіді користувачів
   - Задавайте значення за замовчуванням для некоректного вводу
   - Надсилайте зрозумілі повідомлення про помилки

4. **Відстежуйте ID Запитів**
   - Використовуйте кореляцію між request_id та відповідями
   - Не намагайтеся керувати станом вручну

5. **Проєктуйте для Не-Блокуючого Вводу**
   - Не блоковуйте потоки в очікуванні вводу
   - Використовуйте асинхронні патерни скрізь
   - Підтримуйте одночасне виконання кількох екземплярів робочого процесу

### 📚 Повʼязані Концепції:

- **Проміжне ПЗ для Агентів** - Перехоплення викликів агентів (попередній ноутбук)
- **Управління Станом Робочого Процесу** - Збереження стану між виконаннями
- **Співпраця Багатьох Агентів** - Поєднання людина-в-циклі з командами агентів
- **Архітектури, Що Засновані на Подіях** - Створення реактивних систем з подіями

---

### 🎓 Вітаємо!

Ви опанували робочі процеси людина-в-циклі з Microsoft Agent Framework! Тепер ви вмієте:
- ✅ Призупиняти робочі процеси для збору людського вводу
- ✅ Використовувати RequestInfoExecutor та RequestInfoMessage
- ✅ Працювати з потоковим виконанням через події
- ✅ Створювати користувацьких виконавців з @handler
- ✅ Маршрутизувати робочі процеси на основі людських рішень
- ✅ Будувати інтерактивних AI агентів для співпраці з людьми

**Це критичний патерн для створення надійних, керованих AI систем!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
